# Superstore Sales Data Pipeline

**Tools used:** Databricks, PySpark, Delta Lake and SQL  
**Flow:** Bronze → Silver → Gold

## About this project

I used the Superstore sales CSV to build a small data pipeline in Databricks. The idea was to load the raw file, clean up the fields, check the data, and prepare a final table that could be used for analysis.

The file had **9,994 rows**. While checking the numeric columns, I found **300 rows** where the values could not be converted properly. I kept those rows in a separate table so they could be checked instead of dropping them.

This notebook was built for my Databricks workspace, so the file path and table names may need to be changed before running it elsewhere.

## What the pipeline does

1. **Bronze:** Reads the CSV, tidies the column names and saves the raw data with basic ingestion details.
2. **Silver:** Converts the required fields, checks the records and separates valid rows from rejected ones.
3. **Gold:** Adds a few date-related fields to make the data easier to use for reporting.

## Results from my original run

| Check | Result |
|---|---:|
| Rows loaded into Bronze | 9,994 |
| Rows with failed numeric conversion | 300 |
| Valid rows in Silver | 9,694 |
| Rejected rows | 300 |
| Rows after testing `order_id` + `product_id` as a key | 9,686 |
| Rows kept using `row_id` as the line-item key | 9,694 |
| Rows in Gold | 9,694 |

I cleared the saved notebook outputs before sharing this version. This avoids carrying over old Databricks output and customer-level sample rows. The figures above are from the original run; run the notebook again in Databricks to generate fresh outputs.

## 0. Configuration

The paths and table names below are the ones I used in my workspace. Change them if your catalog, schema or source file is stored somewhere else.

In [ ]:
SOURCE_PATH = "/Volumes/workspace/sales_project/raw_files/sales.csv"

CATALOG = "workspace"
SCHEMA = "sales_project"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_sales"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_sales"
REJECTED_TABLE = f"{CATALOG}.{SCHEMA}.silver_rejected"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_sales"


## 1. Bronze layer — loading the source data

I started by reading the CSV with its header and letting Spark infer the initial data types. I then changed the column names to lowercase with underscores, added the ingestion time and source file name, and saved the result as a Delta table.

I kept this layer close to the source so I could work on type conversion and data checks in the next step.

In [ ]:
from pyspark.sql.functions import (
    col,
    coalesce,
    current_timestamp,
    datediff,
    expr,
    lit,
    month,
    sum as spark_sum,
    when,
    year,
)
from pyspark.sql.types import DoubleType, IntegerType

# Read the source CSV.
df = spark.read.csv(
    SOURCE_PATH,
    header=True,
    inferSchema=True
)

print("Source record count:", df.count())
df.printSchema()


In [ ]:
# Standardize column names: trim, replace spaces/hyphens, and lowercase.
standardized_columns = [
    name.strip().replace(" ", "_").replace("-", "_").lower()
    for name in df.columns
]

df = df.toDF(*standardized_columns)
df.printSchema()


In [ ]:
# Add ingestion metadata and persist the Bronze layer.
df_bronze = (
    df
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", lit("sales.csv"))
)

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_TABLE)
)

bronze = spark.table(BRONZE_TABLE)

print("Bronze record count:", bronze.count())
bronze.printSchema()


## 2. Silver layer — checking and converting the data

My first check using a regular cast showed zero invalid sales values. That result did not reveal the issue, so I tried `try_cast` to see which values could actually be converted.

This check identified 300 rows with conversion problems. When I looked at some of those rows, I noticed that product names containing quotation marks had affected how parts of the CSV were read. Some values had shifted into neighbouring columns.

I kept these records for review instead of removing them from the pipeline.

In [ ]:
# Safely convert numeric columns.
silver = (
    bronze
    .withColumn("sales", expr("try_cast(sales as double)"))
    .withColumn("quantity", expr("try_cast(quantity as int)"))
)

silver.printSchema()


In [ ]:
# Count failed conversions in the numeric columns.
conversion_failures = silver.select(
    spark_sum(when(col("sales").isNull(), 1).otherwise(0)).alias("sales_nulls"),
    spark_sum(when(col("quantity").isNull(), 1).otherwise(0)).alias("quantity_nulls"),
)

display(conversion_failures)
print("Bronze records:", bronze.count())
print("Silver records after type conversion:", silver.count())


## 3. Data-quality checks

Before saving the cleaned data, I added a few checks:

- `order_id` should have a value.
- `sales` should be present and should not be negative.
- `quantity` should be present and greater than zero.
- `ship_date` should not be earlier than `order_date`.

Rows that pass the checks go into the valid dataset. Rows that fail are kept separately. I also treat an unknown (`NULL`) result from a check as invalid.

In [ ]:
valid_rules = (
    col("order_id").isNotNull()
    & col("sales").isNotNull()
    & col("quantity").isNotNull()
    & (col("sales") >= 0)
    & (col("quantity") > 0)
    & (col("ship_date") >= col("order_date"))
)

# Treat NULL rule results as invalid.
silver_flagged = silver.withColumn(
    "is_valid",
    coalesce(valid_rules, lit(False))
)

good = silver_flagged.filter(col("is_valid")).drop("is_valid")
bad = silver_flagged.filter(~col("is_valid")).drop("is_valid")

print("Valid records:", good.count())
print("Rejected records:", bad.count())


In [ ]:
# Review validation-rule violations.
print("Negative sales:", silver.filter(col("sales") < 0).count())
print("Non-positive quantity:", silver.filter(col("quantity") <= 0).count())
print("Ship date before order date:", silver.filter(col("ship_date") < col("order_date")).count())
print("Null order IDs:", silver.filter(col("order_id").isNull()).count())

# Preview a small, non-sensitive selection of rejected fields.
display(
    bad.select(
        "row_id",
        "order_id",
        "product_id",
        "product_name",
        "sales",
        "quantity",
    ).limit(10)
)


## 4. Deduplication — choosing the key

I tested `order_id` and `product_id` together as a composite key. That reduced the valid row count from **9,694 to 9,686**.

After looking at the result, I decided not to use that combination for the final dataset. An order can contain more than one line item for the same product, so those two fields together can remove rows that should remain.

For this dataset, I used `row_id` as the line-item key instead. That kept the valid count at **9,694**. The composite-key check is still in the notebook as a comparison, but it is not used for the final write.

In [ ]:
# Diagnostic only: test the composite key without using it for the final output.
candidate_by_order_product = good.dropDuplicates(["order_id", "product_id"])

print("Valid records before deduplication:", good.count())
print("Records after order_id + product_id deduplication:", candidate_by_order_product.count())

# Use row_id as the line-item key for the final valid dataset.
good = good.dropDuplicates(["row_id"])

print("Valid records after row_id deduplication:", good.count())


## 5. Saving the Silver data

I saved the valid records to the Silver Delta table and the records that failed validation to a separate rejected-records table. This way, the rejected data is still available if I need to investigate it later.

In [ ]:
(
    good.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)

(
    bad.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(REJECTED_TABLE)
)

silver_df = spark.table(SILVER_TABLE)
rejected_df = spark.table(REJECTED_TABLE)

print("Silver table records:", silver_df.count())
print("Rejected table records:", rejected_df.count())
silver_df.printSchema()


## 6. Gold layer — preparing the data for analysis

For the Gold table, I used the cleaned Silver data and added three fields:

- `order_year` — the year from `order_date`
- `order_month` — the month from `order_date`
- `delivery_days` — the number of days between the order date and ship date

I then saved the result as a Delta table for reporting or further analysis.

In [ ]:
gold_sales = (
    silver_df
    .drop("ingestion_time", "source_file")
    .withColumn("order_year", year("order_date"))
    .withColumn("order_month", month("order_date"))
    .withColumn("delivery_days", datediff("ship_date", "order_date"))
)

(
    gold_sales.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLE)
)

gold_df = spark.table(GOLD_TABLE)

print("Gold table records:", gold_df.count())
gold_df.printSchema()

# Preview business fields without displaying customer names or other customer details.
display(
    gold_df.select(
        "order_id",
        "category",
        "sales",
        "quantity",
        "profit",
        "order_year",
        "order_month",
        "delivery_days",
    ).limit(10)
)


## 7. Final results

The original run finished with these row counts:

| Layer / check | Rows |
|---|---:|
| Bronze | 9,994 |
| Valid Silver | 9,694 |
| Rejected records | 300 |
| Gold | 9,694 |

The 300 rejected rows are stored separately. I used `row_id` for the final deduplication because the `order_id` + `product_id` test removed eight rows from the valid dataset.

## Running this notebook

1. Put `sales.csv` in the Volume path set in `SOURCE_PATH`, or update that path.
2. Check that the catalog and schema exist and that your Databricks user can create or overwrite the Delta tables.
3. Run the notebook from the first cell to the last.
4. Check the rejected-records table before using the Gold table.

The notebook creates these tables:

- `bronze_sales`
- `silver_sales`
- `silver_rejected`
- `gold_sales`
